# Инференс

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedShuffleSplit

from catboost import CatBoostClassifier

from genre_ensemble import GenreStackingClassifier, BASE_NAMES

sys.path.insert(0, str(Path.cwd()))

DATA_PATH = Path("../../spotify_data.csv")
DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"
MODELS_DIR = Path("../models")
ENSEMBLE_DIR = MODELS_DIR / "ensemble"
for d in (DATA_DIR, SPLITS_DIR, MODELS_DIR, ENSEMBLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 123
FEATURES = [
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
    "time_signature",
]
EXTRA_COLS = ["popularity", "year", "track_id", "artist_name", "track_name"]

In [2]:
test_df = pl.read_parquet(SPLITS_DIR / "test.parquet").to_pandas()
val_df = pl.read_parquet(SPLITS_DIR / "val.parquet").to_pandas()
classes = np.load(MODELS_DIR / "label_classes.npy", allow_pickle=True).tolist()
label_map = {n: i for i, n in enumerate(classes)}
n_classes = len(classes)


def encode(df):
    return df[FEATURES].copy(), np.array([label_map[g] for g in df["genre"].to_numpy()])


X_test, y_test = encode(test_df)
X_val, y_val = encode(val_df)

single = CatBoostClassifier()
single.load_model(str(MODELS_DIR / "catboost_genre.cbm"))
ensemble = GenreStackingClassifier.load(ENSEMBLE_DIR)
print(f"test={X_test.shape} classes={n_classes}")

test=(173965, 13) classes=82


In [3]:
def eval_model(proba, y, label):
    pred = np.argmax(proba, axis=1)
    return {
        "model": label,
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
        "top1_acc": accuracy_score(y, pred),
        "top3_acc": top_k_accuracy_score(y, proba, k=3, labels=np.arange(n_classes)),
        "top5_acc": top_k_accuracy_score(y, proba, k=5, labels=np.arange(n_classes)),
    }


rows = [eval_model(single.predict_proba(X_test), y_test, "single_catboost")]
for name in BASE_NAMES:
    rows.append(
        eval_model(
            ensemble.bases_[name].predict_proba(X_test[FEATURES]),
            y_test,
            f"base.{name}",
        )
    )
rows.append(
    eval_model(ensemble.predict_proba(X_test, calibrated=False), y_test, "ensemble_raw")
)
rows.append(
    eval_model(ensemble.predict_proba(X_test, calibrated=True), y_test, "ensemble")
)

print(pd.DataFrame(rows).set_index("model").round(4))

                 macro_f1  weighted_f1  top1_acc  top3_acc  top5_acc
model                                                               
single_catboost    0.2350       0.2558    0.2756    0.4702    0.5725
base.lgbm          0.2575       0.2803    0.2947    0.4904    0.5922
base.catboost      0.2292       0.2497    0.2705    0.4629    0.5652
ensemble_raw       0.2575       0.2789    0.2843    0.4663    0.5643
ensemble           0.2648       0.2872    0.2843    0.4654    0.5618


In [4]:
single_pred = np.argmax(single.predict_proba(X_test), axis=1)
ens_pred = ensemble.predict(X_test, calibrated=True)


def class_f1(pred):
    rep = classification_report(
        y_test, pred, target_names=classes, output_dict=True, zero_division=0
    )
    return pd.Series({g: rep[g]["f1-score"] for g in classes})


f1_single = class_f1(single_pred)
f1_ens = class_f1(ens_pred)
train_counts = (
    pl
    .read_parquet(SPLITS_DIR / "train.parquet")
    .to_pandas()["genre"]
    .value_counts()
    .reindex(classes, fill_value=0)
)

delta = pd.DataFrame({
    "genre": classes,
    "train_count": train_counts.values,
    "f1_single": f1_single.values,
    "f1_ensemble": f1_ens.values,
})
delta["delta"] = delta["f1_ensemble"] - delta["f1_single"]
delta_sorted = delta.sort_values("delta", ascending=False)
print("top 15 gainers:")
print(delta_sorted.head(15).to_string(index=False))
print("\ntop 10 regressors:")
print(delta_sorted.tail(10).to_string(index=False))

fig = px.scatter(
    delta,
    x="train_count",
    y="delta",
    hover_data=["genre", "f1_single", "f1_ensemble"],
    log_x=True,
    title="Per-class F1 delta (ensemble - single) vs train count",
)
fig.add_hline(y=0, line_dash="dash")
fig.update_layout(height=520)
fig.show()

top 15 gainers:
        genre  train_count  f1_single  f1_ensemble    delta
          sad         4260   0.230857     0.340845 0.109988
    hardstyle         8694   0.588379     0.690972 0.102594
       groove         9356   0.073817     0.158062 0.084245
      romance         4563   0.341664     0.423569 0.081905
      spanish        13411   0.033920     0.109760 0.075840
        party         6684   0.301573     0.376068 0.074496
        tango        10548   0.578363     0.652608 0.074245
          emo        14591   0.111039     0.184558 0.073519
   deep-house        12527   0.238611     0.311560 0.072949
        sleep        12496   0.587739     0.654403 0.066664
        salsa        12251   0.529834     0.592890 0.063056
        forro        13565   0.448311     0.503735 0.055424
drum-and-bass        10214   0.663403     0.718707 0.055304
       german        11492   0.033704     0.084628 0.050925
         funk        10595   0.093424     0.143934 0.050510

top 10 regressors:
    

In [5]:
worst = (
    delta_sorted.tail(15)["genre"].tolist() + delta_sorted.head(15)["genre"].tolist()
)
keep_idx = [classes.index(g) for g in worst]
mask = np.isin(y_test, keep_idx)
cm = confusion_matrix(y_test[mask], ens_pred[mask], labels=keep_idx)
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig = px.imshow(
    cm_norm,
    x=worst,
    y=worst,
    color_continuous_scale="Viridis",
    labels={"color": "P(pred|true)"},
    title="Confusion matrix (worst 15 regressors + gainers)",
)
fig.update_layout(height=720, width=720, xaxis_tickangle=-60)
fig.show()

In [6]:
shap_idx, _ = next(
    StratifiedShuffleSplit(n_splits=1, train_size=1500, random_state=SEED).split(
        X_test, y_test
    )
)
X_shap = X_test.iloc[shap_idx]

rows = []
for name in BASE_NAMES:
    t0 = time.time()
    sv = ensemble.shap_values_base(X_shap, name)
    abs_mean = np.abs(sv).mean(axis=(0, 1))
    for f, v in zip(FEATURES, abs_mean):
        rows.append({"base": name, "feature": f, "mean_abs_shap": float(v)})
    print(f"shap {name}: {time.time() - t0:.1f}s  shape={sv.shape}")

shap_df = pd.DataFrame(rows)
fig = px.bar(
    shap_df,
    x="mean_abs_shap",
    y="feature",
    color="base",
    barmode="group",
    orientation="h",
    title="SHAP importance per base model",
)
fig.update_layout(height=560, yaxis={"categoryorder": "total ascending"})
fig.show()

shap lgbm: 212.6s  shape=(1500, 82, 13)
shap catboost: 3.1s  shape=(1500, 82, 13)


In [7]:
mw = ensemble.meta_weights()
mw_norm = mw.div(mw.sum(axis=1), axis=0)
print("mean meta influence per base:")
print(mw_norm.mean(axis=0).round(3))
print("\n10 classes most reliant on lgbm:")
print(mw_norm.sort_values("lgbm", ascending=False).head(10).round(3))
print("\n10 classes most reliant on catboost:")
print(mw_norm.sort_values("catboost", ascending=False).head(10).round(3))

mean meta influence per base:
lgbm        0.518
catboost    0.482
dtype: float64

10 classes most reliant on lgbm:
           lgbm  catboost
rock      0.578     0.422
german    0.570     0.430
edm       0.559     0.441
swedish   0.555     0.445
k-pop     0.541     0.459
afrobeat  0.541     0.459
country   0.540     0.460
forro     0.538     0.462
blues     0.538     0.462
dubstep   0.537     0.463

10 classes most reliant on catboost:
                 lgbm  catboost
metalcore       0.483     0.517
sad             0.485     0.515
sertanejo       0.489     0.511
minimal-techno  0.490     0.510
club            0.496     0.504
romance         0.497     0.503
ambient         0.498     0.502
disco           0.500     0.500
breakbeat       0.501     0.499
comedy          0.501     0.499


In [8]:
ens_sv = ensemble.shap_values_ensemble(X_shap, weight_by_meta=True)
print("ensemble shap shape:", ens_sv.shape)

target_genres = [
    g
    for g in ["classical", "death-metal", "hip-hop", "acoustic", "edm"]
    if g in classes
]
records = []
for g in target_genres:
    ci = classes.index(g)
    for f, v in zip(FEATURES, np.abs(ens_sv[:, ci, :]).mean(axis=0)):
        records.append({"genre": g, "feature": f, "mean_abs_shap": float(v)})

fig = px.bar(
    pd.DataFrame(records),
    x="mean_abs_shap",
    y="feature",
    color="genre",
    barmode="group",
    orientation="h",
    title="Ensemble SHAP by genre",
)
fig.update_layout(height=620, yaxis={"categoryorder": "total ascending"})
fig.show()

global_shap = (
    shap_df.groupby("feature")["mean_abs_shap"].mean().sort_values(ascending=False)
)
pl.from_pandas(global_shap.reset_index()).write_parquet(
    DATA_DIR / "shap_global.parquet"
)

ensemble shap shape: (1500, 82, 13)
